In [ ]:
#her kommer en version hvor jeg trækker på min compute_time_day_summaries funktion i stedet for at have det hele i denne fil. Det er lidt mere elegant og genbrugeligt, og så kan jeg også få timelige data med uden at skulle skrive det hele igen.

load_path = r"E:\Mobilize\Thigh_12mo\Thigh_12mo\results"
results_path = r"E:\Mobilize\Thigh_12mo\Thigh_12mo\results\processed"

import os
import sys
from pathlib import Path
import pandas as pd

# Gør analysis-mappen importérbar
sys.path.insert(0, str(Path(r"C:\Users\LAB-ADMIN\Documents\Actimotus2\analysis")))
from activity_summeries import compute_time_day_summaries


def add_mvpa_sedentary(df):
    mvpa_cols = ["fast-walk", "fast_walk", "run", "stairs", "bicycle", "row"]
    sedentary_cols = ["sit", "lie"]

    df = df.copy()
    df["mvpa"] = df[[c for c in mvpa_cols if c in df.columns]].sum(axis=1)
    df["sedentary"] = df[[c for c in sedentary_cols if c in df.columns]].sum(axis=1)
    return df


os.makedirs(results_path, exist_ok=True)

all_results = []
all_hourly = []
csv_files = [f for f in os.listdir(load_path) if f.lower().endswith(".csv")]

for csv_name in csv_files:
    try:
        df = pd.read_csv(
            os.path.join(load_path, csv_name),
            usecols=["datetime", "activity", "steps"],
            dtype={"activity": "category", "steps": "float32"},
            parse_dates=["datetime"],
            comment="#",
            low_memory=False,
        )

        # Brug din fælles summaries-funktion
        daily_exposures, hourly_exposures = compute_time_day_summaries(
            df,
            datetime_col="datetime",
            activity_col="activity",
            steps_col="steps",
        )

        file_stem = os.path.splitext(csv_name)[0]

        daily_exposures = daily_exposures.copy()
        daily_exposures["file"] = file_stem
        all_results.append(daily_exposures)

        hourly_exposures = hourly_exposures.copy()
        hourly_exposures["file"] = file_stem
        all_hourly.append(hourly_exposures)

        print(f"Processed: {csv_name}")

    except Exception as e:
        print(f"FEJL i {csv_name}: {e}")

if all_results:
    combined = pd.concat(all_results, ignore_index=False)

    # Efterfyld kun numeriske kolonner (aktiviteter + total_steps)
    numeric_cols = combined.select_dtypes(include=["number"]).columns
    combined[numeric_cols] = combined[numeric_cols].fillna(0)

    combined = add_mvpa_sedentary(combined)

    # Tilføj weekday kolonne
    combined['weekday'] = combined.index.day_name()

    # Fjern dubletkolonnen non-wear (non-wear_minutes bevares)
    if "non-wear" in combined.columns:
        combined = combined.drop(columns=["non-wear"])

    output_file = os.path.join(results_path, "all_daily_exposures.csv")
    combined[["file", *[c for c in combined.columns if c != "file"]]].to_csv(output_file, encoding="utf-8")
    print(f"\nFærdig! Gemt i: {output_file}")

    if all_hourly:
        combined_hourly = pd.concat(all_hourly, ignore_index=False)

        # Efterfyld kun numeriske kolonner (aktiviteter + total_steps)
        numeric_cols_h = combined_hourly.select_dtypes(include=["number"]).columns
        combined_hourly[numeric_cols_h] = combined_hourly[numeric_cols_h].fillna(0)

        combined_hourly = add_mvpa_sedentary(combined_hourly)

        # Tilføj weekday og hour kolonner
        combined_hourly['weekday'] = combined_hourly.index.day_name()
        combined_hourly['hour'] = combined_hourly.index.hour

        # Fjern dubletkolonnen non-wear (non-wear_minutes bevares)
        if "non-wear" in combined_hourly.columns:
            combined_hourly = combined_hourly.drop(columns=["non-wear"])

        output_file_hourly = os.path.join(results_path, "all_hourly_exposures.csv")
        combined_hourly[["file", *[c for c in combined_hourly.columns if c != "file"]]].to_csv(output_file_hourly, encoding="utf-8")
        print(f"Time data gemt i: {output_file_hourly}")
else:
    print("Ingen filer blev processeret")

Processed: 19681_0002185811.csv
Processed: 19681_0002286811.csv
Processed: 19681_0002337411.csv
Processed: 21348_0002557711.csv
Processed: 21583_0002067811.csv
Processed: 21583_0002416411.csv
Processed: 21607_0003077511.csv
Processed: 21607_0003246511.csv
Processed: 21637_0005118511.csv
Processed: 21637_0005185511.csv
Processed: 21637_0005247311.csv
Processed: 21785_0005108111.csv
Processed: 21785_0005317111.csv
Processed: 21832_0003055211.csv
Processed: 21832_0003366911.csv
Processed: 21883_0003095911.csv
Processed: 21883_0003296911.csv
Processed: 21891_0003225911.csv
Processed: 21891_0003378011.csv
Processed: 21976_0005065311.csv
Processed: 21976_0005136211.csv
Processed: 21976_0005157611.csv
Processed: 21976_0005276711.csv
Processed: 23805_0004147611.csv
Processed: 23805_0004327611.csv
Processed: 23805_0004445811.csv
Processed: 23812_0004037411.csv
Processed: 23812_0004087811.csv
Processed: 23812_0004158511.csv
Processed: 23812_0004176911.csv
Processed: 23821_0004066711.csv
Processe